# Part 4: Recycling Instruction Generation with RAG

Given a waste category, the system finds the matching policy text, a fine-tuned small language model (`flan-t5-small`) writes the recycling instructions from it, and a checker verifies the answer against that text.

Needs `waste_descriptions.csv` and `waste_policy_documents.json` next to this notebook, plus PyTorch, transformers and sentence-transformers (`pip install torch --index-url https://download.pytorch.org/whl/cpu`, then `pip install transformers sentence-transformers textstat`). Creates the folder `waste_rag_model`, which Part 5 loads. Fine-tuning takes about 20 minutes on a CPU (the unseen-category run about 17). Numbers in the Results come from my runs.

## Step 1: Loading Libraries and Checking the Machine

In [ ]:
import os
os.environ["USE_TF"] = "0"          # keep transformers from loading TensorFlow
import re
import json
import time
import copy
import random
import collections
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
print("torch", torch.__version__)

**Result:** Python 3.13 on Windows 11, CPU only, 16.9 GB of RAM (about 10 GB free after closing other notebooks; only 2.2 GB before, which is the real limit for loading models). torch 2.14.1+cpu, transformers 5.18.0, sentence-transformers 6.1.0.

## Step 2: Policy Chunks

Split every policy document by section and tag each chunk with its category.

In [ ]:
desc = pd.read_csv("waste_descriptions.csv")
policies = json.load(open("waste_policy_documents.json"))
header_pattern = re.compile(r"^([A-Za-z][A-Za-z \-/&]+):\s*$", re.M)

def split_sections(text):
    parts = header_pattern.split(text)
    return list(zip(parts[1::2], [b.strip() for b in parts[2::2]]))

chunks = []
for policy in policies:
    for header, body in split_sections(policy["document_text"]):
        categories = policy["categories_covered"]
        match = re.match(r"([A-Z ]+) GUIDELINES$", header)
        if len(categories) > 1:
            if header == "GENERAL REQUIREMENTS":
                categories = ["ALL"]
            elif match:
                own = [c for c in categories if c.upper() == match.group(1).strip()]
                categories = own or categories
        chunks.append({"chunk_id": f"policy{policy['policy_id']}::{header}", "policy_id": policy["policy_id"],
                       "header": header, "category": categories[0],
                       "text": f"{policy['policy_type']} | {header}\n{body}"})
chunk_table = pd.DataFrame(chunks)
print("chunks before filtering:", len(chunk_table))

In [ ]:
# keep the category documents (1-9) and ONE copy of the general rules
own_docs = chunk_table[chunk_table["policy_id"] <= 9]
general_chunk = chunk_table[chunk_table["header"] == "GENERAL REQUIREMENTS"].head(1)

# one disposal card per category, built from the CSV
cards = []
for category, group in desc.groupby("category"):
    instructions = sorted(group["disposal_instruction"].dropna().unique())
    note = group["common_confusion"].dropna().unique()
    text = (f"{category} | Disposal card\nComposition: {group['material_composition'].iloc[0]}\n"
            f"Disposal: " + " ".join(instructions))
    if len(note) > 0:
        text += "\nCommon confusion: " + note[0]
    cards.append({"chunk_id": f"card::{category}", "policy_id": None, "header": "Disposal card",
                  "category": category, "text": text})

rag_corpus = pd.concat([own_docs, general_chunk, pd.DataFrame(cards)], ignore_index=True)
CATEGORIES = sorted(desc["category"].unique())

print("chunks:", len(rag_corpus))
print(rag_corpus["category"].value_counts().to_dict())
print(rag_corpus["text"].str.split().str.len().describe().round(0).to_dict())

**Result:** 67 chunks, 54 kept: 45 policy chunks and 9 disposal cards. Docs 10 to 14 are dropped because every bullet in them is already in the category's own document, and keeping them would risk contradictory placements (window glass). 6 chunks per category (Miscellaneous Trash 5, its document has four sections) plus one general chunk. Chunks are short (median 26 words, maximum 75), so no further splitting is needed.

## Step 3: Embeddings and the Retrieval Index

Two indexes over the 54 chunks: TF-IDF (word matching) and dense MiniLM embeddings (384 numbers per chunk, meaning matching). For 54 chunks an index is just a matrix plus a cosine search.

In [ ]:
chunk_texts = rag_corpus["text"].tolist()
chunk_category = rag_corpus["category"].values

tfidf = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, stop_words="english")
tfidf_matrix = tfidf.fit_transform(chunk_texts)

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
dense_matrix = encoder.encode(chunk_texts, normalize_embeddings=True, show_progress_bar=False)

print("TF-IDF index:", tfidf_matrix.shape)
print("dense index:", dense_matrix.shape)

In [ ]:
def tfidf_scores(question):
    return cosine_similarity(tfidf.transform([question]), tfidf_matrix)[0]

def dense_scores(question):
    return dense_matrix @ encoder.encode([question], normalize_embeddings=True)[0]

def hybrid_scores(question, alpha=0.5):
    a, b = tfidf_scores(question), dense_scores(question)
    a = (a - a.mean()) / (a.std() + 1e-9)
    b = (b - b.mean()) / (b.std() + 1e-9)
    return alpha * a + (1 - alpha) * b           # alpha is the weight of TF-IDF

## Step 4: Compare the Retrieval Methods

A chunk counts as relevant if it belongs to the question's category. 36 category questions ("How to recycle Glass", 4 wordings x 9 categories) and 18 item questions without the category name ("How do I recycle a pizza box?", written by me).

In [ ]:
wordings = ["How to recycle {c}", "How do I dispose of {c} properly?",
            "What is the correct way to recycle {c}?", "{c} recycling rules"]
category_questions = [(w.format(c=c), c) for c in CATEGORIES for w in wordings]

item_questions = [
    ("How do I recycle a pizza box?", "Cardboard"), ("What should I do with shipping boxes?", "Cardboard"),
    ("How do I dispose of banana peels?", "Food Organics"), ("Can I compost coffee grounds and eggshells?", "Food Organics"),
    ("How do I recycle a wine bottle?", "Glass"), ("What do I do with a broken glass jar?", "Glass"),
    ("How do I recycle aluminium cans?", "Metal"), ("Can I recycle an empty aerosol can?", "Metal"),
    ("How do I dispose of old batteries?", "Miscellaneous Trash"), ("How do I dispose of light bulbs?", "Miscellaneous Trash"),
    ("How do I recycle old newspapers?", "Paper"), ("What do I do with shredded documents?", "Paper"),
    ("How do I recycle a plastic bottle?", "Plastic"), ("Can I recycle plastic bags?", "Plastic"),
    ("What should I do with old clothes?", "Textile Trash"), ("How do I dispose of worn towels?", "Textile Trash"),
    ("What do I do with grass clippings?", "Vegetation"), ("How do I dispose of tree branches?", "Vegetation"),
]

In [ ]:
def evaluate_retrieval(scores_function, questions):
    hit1, hit3, hit5, reciprocal = [], [], [], []
    for question, category in questions:
        order = np.argsort(-scores_function(question))
        relevant = chunk_category[order] == category
        hit1.append(relevant[:1].any())
        hit3.append(relevant[:3].any())
        hit5.append(relevant[:5].any())
        first = np.where(relevant)[0]
        reciprocal.append(1 / (first[0] + 1) if len(first) else 0)
    return {"MRR": round(float(np.mean(reciprocal)), 3), "hit@1": round(float(np.mean(hit1)), 3),
            "hit@3": round(float(np.mean(hit3)), 3), "hit@5": round(float(np.mean(hit5)), 3)}

rows = []
methods = {"TF-IDF": tfidf_scores, "dense (MiniLM)": dense_scores, "hybrid 0.5": hybrid_scores}
for method_name, scores_function in methods.items():
    for set_name, questions in [("category questions", category_questions), ("item questions", item_questions)]:
        rows.append({"method": method_name, "questions": set_name, **evaluate_retrieval(scores_function, questions)})
print(pd.DataFrame(rows).to_string(index=False))

**Result:** Category questions: TF-IDF and the hybrid are best (35 of 36 right at rank 1), because the question shares the category name with the chunk. Item questions: dense is best (16 of 18 at rank 1, MRR 0.944, against 12 of 18 and 0.783 for TF-IDF), because linking an item to a category needs meaning. The set is small (one item question is worth 5.6 points) and "relevant" only checks the category.

## Step 5: Choose the Hybrid Weight and Look at the Failures

In [ ]:
rows = []
for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    scores_function = lambda question, a=alpha: hybrid_scores(question, a)
    cat_result = evaluate_retrieval(scores_function, category_questions)
    item_result = evaluate_retrieval(scores_function, item_questions)
    rows.append({"alpha (TF-IDF weight)": alpha, "category MRR": cat_result["MRR"],
                 "category hit@1": cat_result["hit@1"], "item MRR": item_result["MRR"], "item hit@1": item_result["hit@1"]})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
def show_failures(scores_function, questions, name):
    print(name, "- questions whose top result is the wrong category")
    for question, category in questions:
        order = np.argsort(-scores_function(question))
        if chunk_category[order[0]] != category:
            top_three = [rag_corpus["chunk_id"][j] for j in order[:3]]
            print("  ", question, "| expected", category, "->", top_three)

show_failures(tfidf_scores, item_questions, "TF-IDF")
show_failures(dense_scores, item_questions, "dense")

**Result:** alpha = 0.25 is best on both sets: MRR 1.000 on category questions (36 of 36) and 0.944 on item questions (16 of 18), as good as pure dense. It was tuned on the same questions it is scored on, so it is slightly optimistic. TF-IDF fails when generic words ("recycle") match the generic Preparation chunks of Glass and Plastic. "banana peels" fails for both: the word "banana" is in no document (they say "fruit and vegetable scraps"), which is why an item should first be classified into a category. "light bulbs" returns the Glass Non-Acceptable chunk first, which does list light bulbs, so that result is relevant even though the category differs. Decision: hybrid with alpha 0.25, restricted to the requested category when it is known.

## Step 6: Reference Answers

The model needs a target answer per category. We build it mechanically from the sources (collection method, preparation steps, items not to recycle, the CSV disposal sentences, and the confusion note), so nothing is written from memory. Miscellaneous Trash has its own layout.

In [ ]:
def chunk_text(category, header):
    row = rag_corpus[(rag_corpus["category"] == category) & (rag_corpus["header"] == header)]
    if row.empty:
        return ""
    return row["text"].iloc[0].split("\n", 1)[1].strip()

def bullets_of(body):
    return [line[2:].strip() for line in body.splitlines() if line.startswith("- ")]

def prose_of(body):
    lines = [line.strip() for line in body.splitlines() if line.strip() and not line.startswith("- ")]
    return " ".join(lines)

In [ ]:
def reference_answer(category):
    card = chunk_text(category, "Disposal card")
    disposal = card.split("Disposal: ")[1].split("\nCommon confusion:")[0].strip()
    note = ""
    if "Common confusion: " in card:
        note = card.split("Common confusion: ")[1].strip()

    if category == "Miscellaneous Trash":
        cannot = bullets_of(chunk_text(category, "Items That Cannot Be Recycled"))
        special = bullets_of(chunk_text(category, "Special Handling Items"))
        text = (f"{category} cannot be recycled as ordinary material. Cannot be recycled: {'; '.join(cannot)}. "
                f"Special handling: {'; '.join(special)}. {disposal}")
    else:
        collect = prose_of(chunk_text(category, "Collection Method"))
        prepare = bullets_of(chunk_text(category, "Preparation Instructions"))
        avoid = bullets_of(chunk_text(category, "Non-Acceptable Items"))
        text = (f"To recycle {category}: {collect} Before recycling: {'; '.join(prepare)}. "
                f"Do not recycle: {'; '.join(avoid)}. {disposal}")
    if note:
        text += " Note: " + note
    return text.strip()

references = {c: reference_answer(c) for c in CATEGORIES}
print(references["Glass"])
print({c: len(r.split()) for c, r in references.items()})

**Result:** Each reference is 95 to 110 words and every sentence comes from the Glass (or other category's) policy sections or the CSV card. They repeat themselves a little ("Rinse containers" then "Rinse container before recycling") because the policy text and the CSV overlap, which lowers readability. Each reference is built from the same text retrieval returns, so a model could pass by copying; the unseen-category test (Step 18) checks for that.

## Step 7: Training Examples

Input: "Answer the question using only the context. Question: ... Context: ...". Target: the category's reference answer. The context always holds the chunks the reference needs plus 0 to 2 chunks from other categories, shuffled, so the model must pick the right ones. If the needed chunks were sometimes missing, the model would learn to invent.

In [ ]:
os.environ["USE_TF"] = "0"
tok = AutoTokenizer.from_pretrained("google/flan-t5-small")

needed_headers = ["Collection Method", "Preparation Instructions", "Non-Acceptable Items", "Disposal card"]
misc_headers = ["Items That Cannot Be Recycled", "Special Handling Items", "Disposal card"]

def needed_chunks(category):
    headers = misc_headers if category == "Miscellaneous Trash" else needed_headers
    found = []
    for header in headers:
        match = rag_corpus[(rag_corpus["category"] == category) & (rag_corpus["header"] == header)]
        found.append(match.iloc[0])
    return found

def make_input(question, chunk_rows):
    context = " ".join(" ".join(row["text"].split()) for row in chunk_rows)
    return f"Answer the question using only the context. Question: {question} Context: {context}"

In [ ]:
train_wordings = ["How to recycle {c}", "How do I dispose of {c} properly?", "What is the correct way to recycle {c}?",
                  "{c} recycling rules", "Explain how to recycle {c}", "What should I do with {c}?",
                  "Give me instructions for {c}", "How can I get rid of {c} responsibly?"]
val_wordings = ["Tell me how to recycle {c}", "Instructions for recycling {c}"]

def build_examples(categories, wordings, copies):
    examples = []
    for category in categories:
        needed = needed_chunks(category)
        others = rag_corpus[(rag_corpus["category"] != category) & (rag_corpus["category"] != "ALL")]
        for wording in wordings:
            for n_distractors in (0, 1, 2):
                for _ in range(copies):
                    picked = [others.iloc[i] for i in random.sample(range(len(others)), n_distractors)]
                    chunk_rows = needed + picked
                    random.shuffle(chunk_rows)
                    question = wording.format(c=category)
                    examples.append({"category": category, "question": question,
                                     "input": make_input(question, chunk_rows), "target": references[category]})
    return pd.DataFrame(examples)

train_examples = build_examples(CATEGORIES, train_wordings, copies=2)
val_examples = build_examples(CATEGORIES, val_wordings, copies=1)
print("examples (train, validation):", len(train_examples), len(val_examples))

input_lengths = np.array([len(tok(s).input_ids) for s in train_examples["input"]])
target_lengths = np.array([len(tok(s).input_ids) for s in train_examples["target"]])
print("input tokens: mean", round(input_lengths.mean()), "max", input_lengths.max(), "| over 512:", int((input_lengths > 512).sum()))
print("target tokens: mean", round(target_lengths.mean()), "max", target_lengths.max())

**Result:** 432 training and 54 validation examples (8 training wordings, 2 different validation wordings). Inputs are 273 tokens on average, 462 at most, none over the 512 limit; targets are 152 tokens on average, 170 at most.

## Step 8: The Model Before Fine-Tuning

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small").float()
print("parameters:", sum(p.numel() for p in model.parameters()))

example = train_examples[train_examples["category"] == "Glass"].iloc[0]
inputs = tok(example["input"], return_tensors="pt", truncation=True, max_length=512)
model.eval()
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=200, num_beams=1)
print("QUESTION:", example["question"])
print("UNTRAINED ANSWER:", tok.decode(output[0], skip_special_tokens=True))

**Result:** `flan-t5-small` has 76,961,152 parameters. The untrained model answers only "Glass recycling guidelines | Non-Acceptable Items", a heading copied from the context with no instructions. It does not know the answer format, so fine-tuning is needed.

## Step 9: Fine-Tune the Model

A plain PyTorch loop. Padding is excluded from the loss, gradients are clipped, and the weights with the best validation loss are kept (like `EarlyStopping`).

In [ ]:
def encode_batch(inputs, targets):
    batch = tok(inputs, max_length=512, truncation=True, padding=True, return_tensors="pt")
    labels = tok(text_target=targets, max_length=200, truncation=True, padding=True, return_tensors="pt").input_ids
    labels[labels == tok.pad_token_id] = -100          # padding must not count in the loss
    batch["labels"] = labels
    return batch

def make_loader(examples, batch_size, shuffle):
    pairs = list(zip(examples["input"], examples["target"]))
    return DataLoader(pairs, batch_size=batch_size, shuffle=shuffle,
                      collate_fn=lambda b: encode_batch([x[0] for x in b], [x[1] for x in b]))

def average_loss(model, loader):
    model.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            size = batch["input_ids"].size(0)
            total += model(**batch).loss.item() * size
            count += size
    return total / count

In [ ]:
def finetune(train_set, val_set, epochs=3, batch_size=8, lr=3e-4):
    torch.manual_seed(42)
    new_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small").float()
    optimizer = torch.optim.AdamW(new_model.parameters(), lr=lr)
    train_loader = make_loader(train_set, batch_size, True)
    val_loader = make_loader(val_set, batch_size, False)
    best_loss, best_weights, history = float("inf"), None, []

    for epoch in range(1, epochs + 1):
        new_model.train()
        start, running = time.time(), 0.0
        for step_number, batch in enumerate(train_loader, 1):
            loss = new_model(**batch).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(new_model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
            running += loss.item()
            if step_number % 10 == 0:
                print(f"  epoch {epoch} step {step_number}/{len(train_loader)} loss {running / step_number:.4f}", flush=True)
        val_loss = average_loss(new_model, val_loader)
        history.append({"epoch": epoch, "train_loss": round(running / len(train_loader), 4),
                        "val_loss": round(val_loss, 4), "seconds": round(time.time() - start)})
        print(history[-1], flush=True)
        if val_loss < best_loss:
            best_loss, best_weights = val_loss, copy.deepcopy(new_model.state_dict())
    new_model.load_state_dict(best_weights)
    return new_model, pd.DataFrame(history)

In [ ]:
# 24 examples per category (216 in total) keeps the run to about 20 minutes on a CPU
train_small = pd.concat([g.sample(24, random_state=42) for _, g in train_examples.groupby("category")])
print("training examples:", len(train_small), "| validation examples:", len(val_examples))

start = time.time()
final_model, history_table = finetune(train_small, val_examples, epochs=3)
print(history_table.to_string(index=False))
print("total minutes:", round((time.time() - start) / 60, 1))

final_model.save_pretrained("waste_rag_model")
tok.save_pretrained("waste_rag_model")

**Result:** 3 epochs in 20.2 minutes (about 6.8 per epoch). Training loss 0.2996, 0.0347, 0.0185; validation loss 0.0264, 0.0084, 0.0098, so the epoch-2 weights were kept. Losses are very low because the task is mostly restructuring text that sits in the context. Validation uses the same nine categories as training, so it shows robustness to new wordings, not to an unseen category.

## Step 10: The Fine-Tuned Answer for Glass

In [ ]:
final_model.eval()
example = train_examples[train_examples["category"] == "Glass"].iloc[0]
inputs = tok(example["input"], return_tensors="pt", truncation=True, max_length=512)
with torch.no_grad():
    output = final_model.generate(**inputs, max_new_tokens=200, num_beams=1)
answer = tok.decode(output[0], skip_special_tokens=True)
print("FINE-TUNED:", answer)
print("same as the reference:", answer.strip() == example["target"].strip())

**Result:** The model now writes the complete Glass answer, identical to the reference (95 words). Glass was a training category, so this alone does not show it reads the context. The next steps test that.

## Step 11: Does It Read the Context? (Glass)

Two tests. A: change one fact in the context (the collection sentence) to something the model never saw. B: remove the Non-Acceptable Items chunk.

In [ ]:
def generate_answer(trained_model, question, chunk_rows, **options):
    inputs = tok(make_input(question, chunk_rows), return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        output = trained_model.generate(**inputs, max_new_tokens=200, **options)
    return tok.decode(output[0], skip_special_tokens=True)

glass_chunks = needed_chunks("Glass")

# Test A: edit the collection sentence
edited = []
for row in glass_chunks:
    row = row.copy()
    if row["header"] == "Collection Method":
        row["text"] = row["text"].replace("Place in dedicated glass recycling bins.",
                                          "Take glass to the municipal depot on Saturdays.")
    edited.append(row)
answer_a = generate_answer(final_model, "How to recycle Glass", edited)
print("A:", answer_a[:160])
print("new sentence present:", "municipal depot" in answer_a, "| old sentence present:", "dedicated glass recycling bins" in answer_a)

# Test B: remove the Non-Acceptable Items chunk
without = [row for row in glass_chunks if row["header"] != "Non-Acceptable Items"]
answer_b = generate_answer(final_model, "How to recycle Glass", without)
print("\nB:", answer_b)
print("still lists 'Window glass or mirrors':", "Window glass or mirrors" in answer_b)

**Result:** A: the model wrote the new sentence and dropped the old one, so it copies from the context. B: it did not invent a replacement list, but it still wrote the heading "Do not recycle:" with nothing under it, so the next sentence ("Broken glass should be wrapped...") reads as the item not to recycle. It has memorised the layout of the answer.

## Step 12: The Same Two Tests for All Nine Categories

In [ ]:
removed_header = {"Miscellaneous Trash": "Items That Cannot Be Recycled"}
section_label = {"Miscellaneous Trash": "Cannot be recycled:"}

def first_sentence(text):
    sentence = text.split(". ")[0].strip()
    return sentence if sentence.endswith(".") else sentence + "."

rows = []
for category in CATEGORIES:
    question = f"How to recycle {category}"
    needed = needed_chunks(category)

    # Test A: change one fact
    edited, old_fact, new_fact = [], None, None
    for row in needed:
        row = row.copy()
        if category == "Miscellaneous Trash" and row["header"] == "Special Handling Items":
            old_fact = "Batteries (take to dedicated collection points)"
            new_fact = "Batteries (take to the town hall on Mondays)"
            row["text"] = row["text"].replace(old_fact, new_fact)
        elif category != "Miscellaneous Trash" and row["header"] == "Collection Method":
            old_fact = first_sentence(prose_of(chunk_text(category, "Collection Method")))
            new_fact = f"Take {category.lower()} to the municipal depot on Saturdays."
            row["text"] = row["text"].replace(old_fact, new_fact)
        edited.append(row)
    answer_a = generate_answer(final_model, question, edited)

    # Test B: remove the "do not recycle" section
    header = removed_header.get(category, "Non-Acceptable Items")
    items = bullets_of(chunk_text(category, header))
    answer_b = generate_answer(final_model, question, [r for r in needed if r["header"] != header])
    label = section_label.get(category, "Do not recycle:")
    after = answer_b.split(label)[1].strip()[:45] if label in answer_b else "(section missing)"

    rows.append({"category": category, "new fact used": new_fact in answer_a, "old fact gone": old_fact not in answer_a,
                 "invented items": sum(item in answer_b for item in items), "text after heading": after})
print(pd.DataFrame(rows).to_string(index=False))

**Result:** A: 9 of 9 categories used the edited fact and dropped the old one. B: 0 invented items in all 9. But in 8 of 9 the heading "Do not recycle:" was still written, followed by unrelated context text (a disposal sentence or a composition line), which can mislead ("Do not recycle: Plastic bags can be returned to supermarket collection points"). Only Miscellaneous Trash left the empty section out. Word-overlap checks cannot catch this because every sentence is in the context.

## Step 13: Does Retrieval Supply the Chunks the Model Needs?

Retrieve inside the requested category (plus the general chunk) with the hybrid score, for k = 3 to 6, and count how many of the needed chunks are found.

In [ ]:
def get_chunks(question, category, k=6, alpha=0.25):
    scores = hybrid_scores(question, alpha)
    allowed = np.isin(chunk_category, [category, "ALL"])
    scores = np.where(allowed, scores, -np.inf)
    best = np.argsort(-scores)[:k]
    return [rag_corpus.iloc[i] for i in best]

rows = []
for category in CATEGORIES:
    needed_ids = {row["chunk_id"] for row in needed_chunks(category)}
    for k in (3, 4, 5, 6):
        question = f"How to recycle {category}"
        found = get_chunks(question, category, k)
        found_ids = {row["chunk_id"] for row in found}
        n_tokens = len(tok(make_input(question, found)).input_ids)
        rows.append({"category": category, "k": k, "needed found": len(needed_ids & found_ids),
                     "needed": len(needed_ids), "tokens": n_tokens})
table = pd.DataFrame(rows)
table["complete"] = table["needed found"] == table["needed"]
print(table.pivot(index="category", columns="k", values="needed found"))
print(table.groupby("k").agg(share_complete=("complete", "mean"), max_tokens=("tokens", "max")))

**Result:** All needed chunks are found for 0 of 9 categories at k = 3 and k = 4, 4 of 9 at k = 5, and 9 of 9 at k = 6 (longest input 359 tokens, inside the 512 limit). A category has only 6 chunks, so k = 6 is effectively the whole category: with the category given, retrieval is a filter, and ranking matters more when the category is unknown. The model now also sees the category's Acceptable Items and Benefits chunks, which were never in its training context.

## Step 14: The Whole Pipeline on All Nine Categories

In [ ]:
def token_f1(a, b):
    count_a, count_b = collections.Counter(a.lower().split()), collections.Counter(b.lower().split())
    common = sum((count_a & count_b).values())
    if common == 0:
        return 0.0
    precision, recall = common / sum(count_a.values()), common / sum(count_b.values())
    return 2 * precision * recall / (precision + recall)

outputs, rows = {}, []
for category in CATEGORIES:
    question = f"How to recycle {category}"
    answer = generate_answer(final_model, question, get_chunks(question, category, k=6))
    outputs[category] = answer
    reference = references[category]
    extra = sorted(set(answer.lower().split()) - set(reference.lower().split()))[:8]
    missing = sorted(set(reference.lower().split()) - set(answer.lower().split()))[:8]
    rows.append({"category": category, "exact": answer.strip() == reference.strip(),
                 "token F1": round(token_f1(answer, reference), 3), "extra words": extra, "missing words": missing})
print(pd.DataFrame(rows).to_string(index=False))

**Result:** All 9 answers match the reference exactly (token F1 1.0, no extra or missing words), even with the extra Acceptable Items and Benefits chunks in the context. Together with the edit test (9 of 9) and the removal test (0 invented items) this shows the model uses its context. These are categories seen in training, so the unseen-category test is still needed.

## Step 15: Sampling Methods and Parameters

Greedy, beam search and four kinds of random sampling. Trigram support is the share of three-word sequences in the answer that appear in the retrieved text (the references themselves score below 1.0 because of template words like "Before recycling:", so that is the ceiling).

In [ ]:
def trigram_support(answer, context):
    words = answer.lower().split()
    grams = [tuple(words[i:i + 3]) for i in range(len(words) - 2)]
    if not grams:
        return 1.0
    context_words = context.lower().split()
    context_grams = {tuple(context_words[i:i + 3]) for i in range(len(context_words) - 2)}
    return sum(g in context_grams for g in grams) / len(grams)

settings = {
    "greedy": dict(do_sample=False, num_beams=1),
    "beam search (4)": dict(do_sample=False, num_beams=4, early_stopping=True),
    "sampling T=0.7": dict(do_sample=True, temperature=0.7, top_k=0),
    "sampling T=1.0": dict(do_sample=True, temperature=1.0, top_k=0),
    "top-p 0.9, T=1.0": dict(do_sample=True, temperature=1.0, top_p=0.9, top_k=0),
    "top-k 20, T=1.0": dict(do_sample=True, temperature=1.0, top_k=20),
}
seeds = [1, 2]
records = []
for name, options in settings.items():
    for category in CATEGORIES:
        question = f"How to recycle {category}"
        chunk_rows = get_chunks(question, category, k=6)
        context = " ".join(" ".join(row["text"].split()) for row in chunk_rows)
        for seed in (seeds if options["do_sample"] else [0]):
            torch.manual_seed(seed)
            answer = generate_answer(final_model, question, chunk_rows, **options)
            records.append({"setting": name, "exact": answer.strip() == references[category].strip(),
                            "token F1": token_f1(answer, references[category]),
                            "trigram support": trigram_support(answer, context), "words": len(answer.split())})
    print("done:", name, flush=True)

summary = pd.DataFrame(records).groupby("setting", sort=False).mean().round(3)
print(summary)

**Result:** Greedy, beam search, T = 0.7 and top-p 0.9 reproduce the references exactly (trigram support 0.634, the ceiling). Pure sampling at T = 1.0 and top-k 20 each gave one non-identical answer out of 18 (exact 0.944, trigram support 0.629). The model puts almost all its probability on one answer, so settings barely matter. Choice: greedy decoding, which is deterministic, as accurate as the rest, and about 4 times cheaper than beam search. Only 9 categories and 2 seeds.

## Step 16: The Answer That Deviated

In [ ]:
for name in ["sampling T=1.0", "top-k 20, T=1.0"]:
    for category in CATEGORIES:
        question = f"How to recycle {category}"
        chunk_rows = get_chunks(question, category, k=6)
        for seed in seeds:
            torch.manual_seed(seed)
            answer = generate_answer(final_model, question, chunk_rows, **settings[name])
            if answer.strip() != references[category].strip():
                extra = [w for w in answer.split() if w not in set(references[category].split())]
                print(name, "|", category, "| seed", seed)
                print("  extra words:", extra)
                print("  ANSWER:", answer, "\n")

**Result:** In Paper (T = 1.0 and top-k 20, seed 2) the model wrote a second "Do not recycle:" heading and listed under it "Office paper and mail; Newspapers and magazines; Books and catalogs; Paper bags; Wrapping paper; Shredded paper", which are the Paper policy's Acceptable Items. The answer tells a resident not to recycle what the policy accepts. Every word is in the context, so nothing was invented, and word overlap barely moved (trigram support 0.634 to 0.629). A faithfulness check has to know which list belongs under which heading.

## Step 17: The Section-Aware Checker and the Final Function

Three rules: no heading twice; every bullet of the retrieved chunks appears only under the heading of its own source (Acceptable Items bullets must not appear at all); and a heading may not appear if its source chunk was not retrieved. An answer that fails is rebuilt directly from the policy text.

In [ ]:
HEADINGS = ["Before recycling:", "Do not recycle:", "Cannot be recycled:", "Special handling:", "Note:"]
SOURCE_HEADING = {"Preparation Instructions": "Before recycling:", "Non-Acceptable Items": "Do not recycle:",
                  "Items That Cannot Be Recycled": "Cannot be recycled:", "Special Handling Items": "Special handling:"}

def check_answer(answer, chunk_rows):
    problems = []
    for heading in HEADINGS:                                       # rule 1
        if answer.count(heading) > 1:
            problems.append(f"heading repeated: {heading}")

    positions = sorted((m.start(), h) for h in HEADINGS for m in re.finditer(re.escape(h), answer))
    def heading_before(index):
        earlier = [h for p, h in positions if p < index]
        return earlier[-1] if earlier else None

    for row in chunk_rows:                                         # rule 2
        expected = SOURCE_HEADING.get(row["header"])
        for bullet in bullets_of(row["text"].split("\n", 1)[1]):
            for match in re.finditer(re.escape(bullet), answer):
                found_under = heading_before(match.start())
                if expected is None:
                    problems.append(f"'{bullet}' (from {row['header']}) should not appear")
                elif found_under != expected:
                    problems.append(f"'{bullet}' is under '{found_under}', expected '{expected}'")

    for heading in HEADINGS[:4]:                                   # rule 3
        has_source = any(SOURCE_HEADING.get(row["header"]) == heading for row in chunk_rows)
        if heading in answer and not has_source:
            problems.append(f"{heading} written but its source chunk was not retrieved")
    return problems

In [ ]:
def generate_instructions(category, question=None, k=6):
    matches = [c for c in CATEGORIES if c.lower() == str(category).strip().lower()]
    if not matches:
        return {"category": category, "answer": None, "status": "unknown category", "problems": [], "sources": []}
    category = matches[0]
    question = question or f"How to recycle {category}"
    chunk_rows = get_chunks(question, category, k)
    raw = generate_answer(final_model, question, chunk_rows)             # greedy decoding
    problems = check_answer(raw, chunk_rows)
    if problems:
        answer, status = reference_answer(category), "fallback (failed the check; rebuilt from the sources)"
    else:
        answer, status = raw, "ok"
    return {"category": category, "answer": answer, "status": status, "problems": problems,
            "sources": [row["chunk_id"] for row in chunk_rows], "raw_model_output": raw}

for category in CATEGORIES:
    print(category, "->", generate_instructions(category)["status"])
print(generate_instructions("glass")["status"])
print(generate_instructions("Furniture")["status"])

In [ ]:
# the inverted Paper answer must be caught and replaced
paper_bad = ("To recycle Paper: Place in paper recycling bins or mixed recycling where accepted. Before recycling: Keep paper dry and clean; "
             "Remove plastic wrappings, windows, or bindings; Shred confidential documents. Do not recycle: Food-soiled paper; Waxed paper; "
             "Paper towels and napkins; Thermal receipt paper; Paper with plastic coatings. Do not recycle: Office paper and mail; "
             "Newspapers and magazines; Books and catalogs; Paper bags; Wrapping paper (non-metallic); Shredded paper (in paper bags). "
             "Ensure it is clean and dry before recycling.")
paper_chunks = get_chunks("How to recycle Paper", "Paper", 6)
problems = check_answer(paper_bad, paper_chunks)
print(len(problems), "problems, for example:", problems[:2])

**Result:** All 9 categories give status "ok", lowercase "glass" is accepted, and "Furniture" returns an unknown-category message. The inverted Paper answer is caught (repeated heading, Acceptable Items under "Do not recycle:") and replaced by the rebuilt answer. The checker does not look at the collection sentence, the card sentences or the final note, and it cannot detect a missing section. Honest note: the references are templated, so the rebuild alone gives the right answers for this data; the model's value is that it follows edited facts and does not invent, with its occasional failures caught.

## Step 18: Unseen Category: Paper Held Out

Train a new model from scratch on the eight other categories, then ask it for Paper with Paper's real retrieved chunks.

In [ ]:
train_heldout = train_small[train_small["category"] != "Paper"]
val_heldout = val_examples[val_examples["category"] != "Paper"]
print(len(train_heldout), len(val_heldout), "| Paper in training:", (train_heldout["category"] == "Paper").any())

start = time.time()
heldout_model, heldout_history = finetune(train_heldout, val_heldout, epochs=3)
print(heldout_history.to_string(index=False))
print("minutes:", round((time.time() - start) / 60, 1))

In [ ]:
question = "How to recycle Paper"
paper_chunks = get_chunks(question, "Paper", k=6)
answer = generate_answer(heldout_model, question, paper_chunks)
print("exact match:", answer.strip() == references["Paper"].strip(), "| token F1:", round(token_f1(answer, references["Paper"]), 3))
print("checker problems:", check_answer(answer, paper_chunks))
print("\nANSWER:", answer)

# how much Paper text did training contain?
has_paper_chunk = (train_heldout["input"].str.contains("Paper Recycling Guidelines") |
                   train_heldout["input"].str.contains("Paper | Disposal card", regex=False))
print("\ntraining inputs containing a Paper chunk:", int(has_paper_chunk.sum()), "of", len(train_heldout))
print("training targets that are a Paper answer:", int(train_heldout["target"].str.contains("To recycle Paper:", regex=False).sum()))

**Result:** 17.4 minutes; validation loss 0.0285, 0.0021, 0.0029. The new model wrote the complete correct Paper answer (exact match, token F1 1.0, no checker problems) without ever being trained to produce one. 17 of the 192 training inputs showed a Paper chunk as a distractor, but 0 targets were Paper answers, and the loss is computed only on the answer, so it never learned to write Paper text. Caveats: one category, one run; Paper shares its five-section layout with seven others. A category with a different layout (Miscellaneous Trash) would be a harder test and was not run.

## Step 19: Readability

In [ ]:
# pip install textstat    (once, if missing)
import textstat

def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

def near_duplicate_pairs(text, threshold=0.5):
    word_sets = [set(re.findall(r"[a-z]+", s.lower())) for s in split_sentences(text)]
    pairs = 0
    for i in range(len(word_sets)):
        for j in range(i + 1, len(word_sets)):
            union = word_sets[i] | word_sets[j]
            if union and len(word_sets[i] & word_sets[j]) / len(union) >= threshold:
                pairs += 1
    return pairs

rows = []
for category in CATEGORIES:
    answer = outputs[category]
    sentences = split_sentences(answer)
    rows.append({"category": category, "words": len(answer.split()), "sentences": len(sentences),
                 "words per sentence": round(len(answer.split()) / len(sentences), 1),
                 "Flesch ease": round(textstat.flesch_reading_ease(answer), 1),
                 "grade level": round(textstat.flesch_kincaid_grade(answer), 1),
                 "near-duplicate pairs": near_duplicate_pairs(answer)})
readability = pd.DataFrame(rows)
print(readability.to_string(index=False))
print(readability[["words per sentence", "Flesch ease", "grade level", "near-duplicate pairs"]].mean().round(2).to_dict())

**Result:** Average Flesch reading ease 39.9 (difficult; plain English is 60 or more), grade level 10.3, 10.5 words per sentence. Sentences are short, so the low score comes from long words (recycling, containers, hazardous). Miscellaneous Trash is hardest (11.1, grade 14.4); Food Organics easiest (54.2). Only one near-duplicate sentence pair. Flesch is built for prose, not lists of nouns, so this is a rough guide, and the model cannot be more readable than the policy text it copies.

## Summary

- **System:** category in, hybrid retrieval (TF-IDF weight 0.25 plus MiniLM) inside the category's 6 chunks, fine-tuned `flan-t5-small` with greedy decoding, section-aware checker, rebuild from the sources if it fails. The function is `generate_instructions(category)`.
- **Evidence it uses the documents:** all 9 references reproduced; the edited fact was used in 9 of 9; 0 invented items when a section was removed; a model trained without Paper still wrote the exact Paper answer.
- **What went wrong:** with a section missing the model fills the slot with misplaced text (8 of 9), and under sampling it once listed Paper's accepted items under "Do not recycle:". Word overlap could not see it; the checker does.
- **Limitations:** the references are templated, so the model's value is robustness, not capability; one held-out category sharing a layout with seven others; a small hand-written retrieval set; the checker misses the collection sentence, card sentences and the note; readability is inherited from the policy text; single runs on a CPU.